# TradeSense · Component 03 · Why does BTC volume_acceleration differ on 2024-06-01 and 07-01? (diagnosis, read only)

Notebook 23 showed that the rebuild from raw trades equals the stored Layer 1 for five features (99.9 to 100 percent) and for ETH volume_acceleration, but BTC volume_acceleration matched only 76.9 percent on 06-01 and 97.3 percent on 07-01. This notebook finds the reason: it compares the stored and rebuilt volume bar by bar, locates the mismatching bars (hour, floor active or not), and tests the candidate rules (window length, ddof, floor, previous-day carry). CPU, about 2 minutes. Read only.

In [ ]:
import os, json, glob
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); OUT = BASE/'layer1_rebuild_v1'
FLOOR = {'BTCUSDT': 2.5399, 'ETHUSDT': 27.7658}
DAYS = [date(2024,6,1), date(2024,7,1)]
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def stored(s):
    p = sorted(glob.glob(str(BASE/'**'/f'{s.lower()}_l1_feature_matrix_90d.parquet'), recursive=True)); p = [x for x in p if 'backup' not in x.lower()][0]
    d = pd.read_parquet(p);
    if 'datetime' in d.columns: d = d.set_index('datetime')
    d.index = pd.to_datetime(d.index, utc=True); return d
def rebuilt(s):
    b = pd.concat([pd.read_parquet(p) for p in sorted((OUT/'bars').glob(f'{s}_*_bars.parquet'))]).sort_index()
    return b[~b.index.duplicated(keep='first')]
def va(v, win=15, ddof=1, floor=0.0, incl=False, mask=15):
    s = pd.Series(v); w = s if incl else s.shift(1)
    mu = w.rolling(win).mean().to_numpy(); sd = w.rolling(win).std(ddof=ddof).to_numpy()
    out = (v-mu)/np.maximum(sd, floor); out[:mask] = 0.0; return out

Mounted at /content/drive


## Diagnosis

In [ ]:
for s in ['BTCUSDT', 'ETHUSDT']:
    ex, rb = stored(s), rebuilt(s)
    say(f'===== {s}: stored columns {list(ex.columns)[:14]}')
    for d in DAYS:
        t0 = pd.Timestamp(d, tz='UTC'); sl = slice(t0, t0+pd.Timedelta('23h59m'))
        a, e = rb.loc[sl], ex.loc[sl]
        if len(a) != 1440 or len(e) != 1440: say(f'{d}: lengths rebuilt {len(a)} stored {len(e)} -> skip'); continue
        vcol = 'volume' if 'volume' in e.columns else None
        # 1. is the stored VOLUME the same as the rebuilt volume ?
        if vcol:
            dv = np.abs(a.volume.to_numpy()-e[vcol].to_numpy()); bad = dv > 1e-9*np.maximum(1, e[vcol].to_numpy())
            say(f'{d}: volume differs on {bad.sum()} of 1440 bars (max abs {dv.max():.6g})')
        # 2. which bars of volume_acceleration differ ?
        x, y = a.index, None
        full = pd.concat([rb.loc[:t0-pd.Timedelta('1min')].tail(60), a])          # previous bars for the window
        vfull = full.volume.to_numpy()
        reb = va(vfull, floor=FLOOR[s], mask=0)[-1440:]                               # contract rule on rebuilt volume, previous-day carry
        sto = e['volume_acceleration'].to_numpy()
        diff = np.abs(reb-sto) > 1e-6*np.maximum(1, np.abs(sto)); diff[:15] = False
        say(f'{d}: volume_acceleration mismatching bars {diff.sum()} ; by 4-hour block: ' + str([int(diff[i:i+240].sum()) for i in range(0,1440,240)]))
        w = pd.Series(vfull).shift(1).rolling(15).std(ddof=1).to_numpy()[-1440:]
        fl = w < FLOOR[s]
        say(f'    floor active on {fl.sum()} bars; mismatches where floor active {int((diff&fl).sum())} / not active {int((diff&~fl).sum())}')
        idx = np.where(diff)[0][:6]
        for i in idx: say(f'    bar {i:4d} {a.index[i]:%H:%M}: stored {sto[i]:9.4f} rebuilt {reb[i]:9.4f} vol {a.volume.iloc[i]:9.3f} rolling-std {w[i]:7.3f}')
        # 3. candidate rules applied to the STORED volume column: which reproduces the stored vol_acc ?
        if vcol:
            vs = ex[vcol].to_numpy(); pos = ex.index.get_loc(a.index[0])
            for name, kw in [('contract (win15, ddof1, floor)', {}), ('ddof=0', {'ddof': 0}), ('window 14', {'win': 14}), ('window 16', {'win': 16}),
                             ('includes current bar', {'incl': True}), ('no floor', {'floor': 0.0})]:
                k = dict(floor=FLOOR[s]); k.update(kw)
                r = va(vs, **k)[pos:pos+1440]; ok = np.isclose(r[15:], sto[15:], rtol=1e-6, atol=1e-9).mean()
                say(f'    rule on stored volume: {name:36s} equals stored in {ok*100:6.2f}% of bars')
say('')
say('Reading: if the stored volume equals the rebuilt volume and the contract rule reproduces the stored column, the difference comes from the rebuilt neighbours (previous day) and the mismatch hours show where. '
    'If volume itself differs on some bars, the stored Layer 1 contains a data difference on those minutes (shown with their hours).')

===== BTCUSDT: stored columns ['open', 'high', 'low', 'close', 'volume', 'trades_count', 'log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
2024-06-01: volume differs on 0 of 1440 bars (max abs 3.55271e-15)
2024-06-01: volume_acceleration mismatching bars 284 ; by 4-hour block: [2, 58, 41, 33, 100, 50]
    floor active on 329 bars; mismatches where floor active 284 / not active 0
    bar   80 01:20: stored   -0.6800 rebuilt   -0.6800 vol     4.639 rolling-std   2.308
    bar   81 01:21: stored    8.4355 rebuilt    8.4356 vol    27.475 rolling-std   2.185
    bar  271 04:31: stored   -0.2102 rebuilt   -0.2102 vol     2.814 rolling-std   2.077
    bar  272 04:32: stored    0.2851 rebuilt    0.2851 vol     4.013 rolling-std   2.079
    bar  273 04:33: stored    0.3787 rebuilt    0.3787 vol     4.245 rolling-std   2.076
    bar  274 04:34: stored   -0.2473 rebuilt   -0.2473 vol     2.804 rolling-std   2.058
    rule o

In [ ]:
OUT.mkdir(exist_ok=True); (OUT/'volacc_diagnosis.txt').write_text('\n'.join(LINES)); print('\nsaved', OUT/'volacc_diagnosis.txt')


saved /content/drive/MyDrive/research datasets/layer1_rebuild_v1/volacc_diagnosis.txt
